# Interactive Serve Viewer

This notebook provides two interactive features for tennis serve analysis:

## Features

**1. Interactive Point Selection** — Click to select calibration points (P1, P2) and ball position on a video frame.

**2. Video Playback with Overlays** — Play/scrub through video frames with:
- Yellow calibration line between P1 → P2
- Green/red dots at calibration points
- Blue X at current tracked ball position
- Cyan trajectory trail (fading history)
- Speed label (current km/h)
- Frame number and timestamp

## Instructions

**For Point Selection:**
1. Run the configuration and imports cells
2. When the frame appears, click two calibration points (green = P1, red = P2)
3. Then click the ball position (blue X)
4. Points are stored and used for tracking

**For Video Playback:**
1. After running tracking, use the play button or slider to navigate frames
2. Overlays update in real-time
3. Speed profile chart shows velocity trend

## Configuration

Set your video path and tracking parameters below.

In [ ]:
# === VIDEO CONFIGURATION ===
VIDEO_PATH = "../IMG_9259.MOV"  # Path to your serve video

# === CALIBRATION ===
REAL_DISTANCE = 1.83  # Known real-world distance between calibration points (meters)

# === TRACKING PARAMETERS ===
TEMPLATE_SIZE = 13     # Template window size in pixels
SEARCH_RADIUS = 100    # Search radius around previous ball position
MAX_FRAMES = 100       # Maximum frames to track (None = until end)
SMOOTHING_WINDOW = 3   # Velocity smoothing window (1 = no smoothing)

# === FRAME SELECTION ===
START_FRAME = 550      # Frame to start tracking (post-impact)

# Print configuration
print("Configuration loaded:")
print(f"  Video: {VIDEO_PATH}")
print(f"  Real distance: {REAL_DISTANCE} m")
print(f"  Template size: {TEMPLATE_SIZE} px")
print(f"  Search radius: {SEARCH_RADIUS} px")
print(f"  Max frames: {MAX_FRAMES}")
print(f"  Smoothing window: {SMOOTHING_WINDOW}")
print(f"  Start frame: {START_FRAME}")

## Imports + Video Load

Import required libraries and load video metadata.

In [ ]:
import sys
sys.path.insert(0, "..")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from serve_analyzer.analysis import (
    compute_scale_factor,
    compute_velocity_series,
    track_ball_template,
    get_video_info,
)

# Load video metadata
print("Loading video info...")
video_info = get_video_info(VIDEO_PATH)

print(f"  FPS: {video_info['fps']:.2f}")
print(f"  Resolution: {video_info['width']}x{video_info['height']}")
print(f"  Total frames: {video_info['frame_count']}")
print(f"  Duration: {video_info['duration_sec']:.2f} seconds")

## Step 1: Interactive Point Selection

**Instructions:**
1. Run this cell to display the start frame
2. Click **two calibration points** (you'll be prompted for each)
3. Click **one ball position** (initial tracking point)
4. Points will be drawn on the frame and stored in variables

In [ ]:
%matplotlib widget
# Read the start frame
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)
ret, frame = cap.read()
cap.release()

if not ret:
    raise IOError(f"Cannot read frame {START_FRAME}")

# Convert BGR to RGB for matplotlib display
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# Display frame
fig, ax = plt.subplots(figsize=(20, 12))
ax.imshow(frame_rgb)
ax.set_title(f"Frame {START_FRAME} - Click calibration points", fontsize=14)
ax.axis('off')
fig.tight_layout()

print(f"\n{'='*60}")
print("CLICK SELECTION MODE")
print(f"{'='*60}")
print("Step 1: Click TWO calibration points (P1, P2)")
print("  - Choose two points with a known real-world distance")
print("  - Green dot = P1, Red dot = P2, Yellow line = ruler")
print("\n  💡 TIP: You can use the toolbar on the left of the image")
print("         to Zoom (magnifying glass) or Pan (arrows) BEFORE")
print("         clicking your points. Unselect the tool to click.")
print(f"{'='*60}\n")

# Get calibration points (2 clicks)
calibration_points = plt.ginput(n=2, timeout=0)

if len(calibration_points) < 2:
    raise ValueError("Need exactly 2 calibration points")

CAL_P1 = (int(calibration_points[0][0]), int(calibration_points[0][1]))
CAL_P2 = (int(calibration_points[1][0]), int(calibration_points[1][1]))

# Draw calibration points and line
ax.plot(CAL_P1[0], CAL_P1[1], 'go', markersize=10, label='P1')
ax.plot(CAL_P2[0], CAL_P2[1], 'ro', markersize=10, label='P2')
ax.plot([CAL_P1[0], CAL_P2[0]], [CAL_P1[1], CAL_P2[1]], 'y-', linewidth=2, label='Calibration')

# Calculate pixel distance
pixel_dist = np.sqrt((CAL_P2[0] - CAL_P1[0])**2 + (CAL_P2[1] - CAL_P1[1])**2)
mid_x = (CAL_P1[0] + CAL_P2[0]) // 2
mid_y = (CAL_P1[1] + CAL_P2[1]) // 2
ax.text(mid_x, mid_y - 20, f"{pixel_dist:.1f} px", color='yellow', fontsize=12, 
        ha='center', weight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

ax.legend(loc='upper right')
plt.draw()

print("Calibration points selected:")
print(f"  P1 (green): {CAL_P1}")
print(f"  P2 (red):   {CAL_P2}")
print(f"  Pixel distance: {pixel_dist:.1f} px")
print(f"\n{'='*60}")
print("Step 2: Click the BALL position (initial tracking point)")
print("\n  💡 TIP: You can use the toolbar on the left of the image")
print("         to Zoom (magnifying glass) or Pan (arrows) BEFORE")
print("         clicking your point. Unselect the tool to click.")
print(f"{'='*60}\n")

# Get ball position (1 click)
ball_point = plt.ginput(n=1, timeout=0)

if len(ball_point) < 1:
    raise ValueError("Need ball position")

BALL_POS = (int(ball_point[0][0]), int(ball_point[0][1]))

# Draw ball position
ax.plot(BALL_POS[0], BALL_POS[1], 'bX', markersize=15, markeredgewidth=3, label='BALL')
ax.text(BALL_POS[0] + 20, BALL_POS[1], "BALL", color='cyan', fontsize=12, 
        weight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

ax.legend(loc='upper right')
plt.draw()

print("Ball position selected:")
print(f"  BALL (blue X): {BALL_POS}")
print(f"\n{'='*60}")
print("Point selection complete!")
print(f"{'='*60}\n")

## Step 2: Track Ball

Run template-based ball tracking from the selected starting point.

In [ ]:
# Compute scale factor
scale_factor = compute_scale_factor(CAL_P1, CAL_P2, REAL_DISTANCE)
pixel_dist = np.sqrt((CAL_P2[0] - CAL_P1[0])**2 + (CAL_P2[1] - CAL_P1[1])**2)

print("Scale computation:")
print(f"  Pixel distance: {pixel_dist:.1f} px")
print(f"  Real distance:  {REAL_DISTANCE} m")
print(f"  Scale factor:   {scale_factor:.6f} m/pixel")
print()

# Track ball
print(f"Tracking ball starting at frame {START_FRAME}...")
print(f"  Initial center: {BALL_POS}")
print(f"  Template size: {TEMPLATE_SIZE}, Search radius: {SEARCH_RADIUS}")

centers = track_ball_template(
    video_path=VIDEO_PATH,
    start_frame=START_FRAME,
    initial_center=BALL_POS,
    template_size=TEMPLATE_SIZE,
    search_radius=SEARCH_RADIUS,
    max_frames=MAX_FRAMES,
)

fps = video_info['fps']

# Compute velocity series
speeds_mps, speeds_kmh, stats = compute_velocity_series(
    centers=centers,
    fps=fps,
    scale_factor=scale_factor,
    smoothing_window=SMOOTHING_WINDOW,
)

print(f"\n{'='*60}")
print("TRACKING COMPLETE")
print(f"{'='*60}")
print(f"Frames tracked:   {stats['frame_count']}")
print(f"Duration:         {stats['duration_sec']:.3f} seconds")
print(f"Max speed:        {stats['max_kmh']:.1f} km/h ({stats['max_mps']:.2f} m/s)")
print(f"Mean speed:       {stats['mean_kmh']:.1f} km/h ({stats['mean_mps']:.2f} m/s)")
print(f"Median speed:     {stats['median_kmh']:.1f} km/h ({stats['median_mps']:.2f} m/s)")
print(f"{'='*60}\n")

## Step 3: Video Playback with Overlays

Interactive video player with real-time overlay rendering.

**Controls:**
- **Play button**: Start/pause playback
- **Slider**: Scrub through frames manually
- **Speed**: Adjust playback speed (frames per second)

**Overlays:**
- Yellow line: Calibration ruler (P1 → P2)
- Green/Red dots: Calibration points
- Blue X: Current ball position
- Cyan trail: Ball trajectory (last 10 positions)
- White text: Frame number, timestamp, speed

In [ ]:
# === OVERLAY HELPER FUNCTION ===
def draw_overlays(frame, frame_idx, cal_p1, cal_p2, centers, speeds_kmh, trail_length=10):
    """
    Draw all overlays on a frame.
    
    Args:
        frame: BGR frame from video
        frame_idx: Current frame index (relative to tracking start)
        cal_p1: Calibration point 1 (x, y)
        cal_p2: Calibration point 2 (x, y)
        centers: List of tracked ball centers
        speeds_kmh: Array of speeds in km/h
        trail_length: Number of trail positions to show
    
    Returns:
        Frame with overlays drawn (BGR)
    """
    # Make a copy to avoid modifying original
    overlay_frame = frame.copy()
    
    # Draw calibration line (yellow)
    cv2.line(overlay_frame, cal_p1, cal_p2, (0, 255, 255), 2)
    
    # Draw calibration points (green = P1, red = P2)
    cv2.circle(overlay_frame, cal_p1, 8, (0, 255, 0), -1)
    cv2.circle(overlay_frame, cal_p2, 8, (0, 0, 255), -1)
    
    # Add P1/P2 labels
    cv2.putText(overlay_frame, "P1", (cal_p1[0] + 10, cal_p1[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(overlay_frame, "P2", (cal_p2[0] + 10, cal_p2[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # Draw ball position and trail if within tracked range
    if 0 <= frame_idx < len(centers):
        # Draw trajectory trail (cyan, fading)
        trail_start = max(0, frame_idx - trail_length)
        for i in range(trail_start, frame_idx):
            alpha = (i - trail_start + 1) / (frame_idx - trail_start + 1)
            color = (int(255 * alpha), int(255 * alpha), 0)  # Cyan with alpha
            cv2.circle(overlay_frame, 
                      (int(centers[i][0]), int(centers[i][1])), 
                      3, color, -1)
        
        # Draw current ball position (blue X)
        ball_x, ball_y = int(centers[frame_idx][0]), int(centers[frame_idx][1])
        size = 12
        thickness = 3
        cv2.line(overlay_frame, (ball_x - size, ball_y - size), 
                 (ball_x + size, ball_y + size), (255, 0, 0), thickness)
        cv2.line(overlay_frame, (ball_x - size, ball_y + size), 
                 (ball_x + size, ball_y - size), (255, 0, 0), thickness)
        
        # Draw speed label
        if frame_idx < len(speeds_kmh):
            speed_text = f"{speeds_kmh[frame_idx]:.1f} km/h"
            cv2.putText(overlay_frame, speed_text, (ball_x + 20, ball_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    
    # Draw frame info (top-left corner)
    actual_frame = START_FRAME + frame_idx
    timestamp = actual_frame / fps
    info_text = f"Frame: {actual_frame} | Time: {timestamp:.2f}s"
    cv2.putText(overlay_frame, info_text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    
    return overlay_frame


# === VIDEO CAPTURE SETUP ===
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f"Cannot open video: {VIDEO_PATH}")

# === WIDGET SETUP ===
# Image widget for displaying frames
image_widget = widgets.Image(
    format='png',
    width=video_info['width'] // 3,  # Scale down for display
    height=video_info['height'] // 3,
)

# Play button
play_widget = widgets.Play(
    value=0,
    min=0,
    max=len(centers) - 1,
    step=1,
    interval=100,  # 100ms between frames (10 fps default)
    description="Play",
    disabled=False
)

# Frame slider
slider_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=len(centers) - 1,
    step=1,
    description='Frame:',
    continuous_update=True,
    readout=True,
    readout_format='d'
)

# Speed control
speed_widget = widgets.IntSlider(
    value=100,
    min=20,
    max=500,
    step=20,
    description='Interval (ms):',
)

# Link widgets
widgets.jslink((play_widget, 'value'), (slider_widget, 'value'))

# Update playback speed
def update_speed(change):
    play_widget.interval = change['new']

speed_widget.observe(update_speed, names='value')


# === FRAME UPDATE FUNCTION ===
def update_frame(frame_idx):
    """Update the displayed frame with overlays."""
    # Seek to actual frame
    actual_frame = START_FRAME + frame_idx
    cap.set(cv2.CAP_PROP_POS_FRAMES, actual_frame)
    ret, frame = cap.read()
    
    if not ret:
        return
    
    # Draw overlays
    overlay_frame = draw_overlays(
        frame, frame_idx, CAL_P1, CAL_P2, centers, speeds_kmh, trail_length=10
    )
    
    # Encode to PNG bytes for widget
    _, buffer = cv2.imencode('.png', overlay_frame)
    image_widget.value = buffer.tobytes()


# === CALLBACK FOR WIDGET CHANGES ===
def on_frame_change(change):
    """Handle frame slider/play changes."""
    update_frame(change['new'])

slider_widget.observe(on_frame_change, names='value')


# === DISPLAY INITIAL FRAME ===
update_frame(0)

# === LAYOUT ===
controls = widgets.HBox([play_widget, slider_widget, speed_widget])
layout = widgets.VBox([image_widget, controls])

display(layout)

print("\n" + "="*60)
print("VIDEO PLAYER READY")
print("="*60)
print("Controls:")
print("  - Click Play button to start/pause")
print("  - Drag slider to scrub through frames")
print("  - Adjust interval to change playback speed")
print("="*60)

## Step 4: Speed Profile

Plot the speed profile across all tracked frames. This shows how the ball velocity changes over time (typically accelerating through impact, then decelerating due to air resistance).

In [ ]:
%matplotlib inline

frames = range(len(speeds_kmh))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(frames, speeds_kmh, marker='o', linewidth=1.5, markersize=4, color='#2E86AB')
ax.set_xlabel("Frame (relative to start)", fontsize=12)
ax.set_ylabel("Speed (km/h)", fontsize=12)
ax.set_title("Post-Impact Ball Speed Profile", fontsize=14, weight='bold')
ax.grid(True, alpha=0.3)

# Mark max speed
max_idx = int(np.argmax(speeds_kmh))
ax.plot(max_idx, speeds_kmh[max_idx], 'ro', markersize=10, label='Max speed')
ax.annotate(
    f"Max: {speeds_kmh[max_idx]:.1f} km/h",
    xy=(max_idx, speeds_kmh[max_idx]),
    xytext=(max_idx + 3, speeds_kmh[max_idx] + 2),
    arrowprops=dict(arrowstyle="->", color='red', lw=2),
    color='red',
    fontsize=11,
    weight='bold'
)

# Add stats box
stats_text = (
    f"Max: {stats['max_kmh']:.1f} km/h\n"
    f"Mean: {stats['mean_kmh']:.1f} km/h\n"
    f"Median: {stats['median_kmh']:.1f} km/h"
)
ax.text(0.98, 0.97, stats_text, transform=ax.transAxes, 
        verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
        fontsize=10, family='monospace')

ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print("\nSpeed profile plot generated.")
print(f"Max speed: {stats['max_kmh']:.1f} km/h at frame {max_idx}")

## Cleanup

Release video resources when done.

In [ ]:
# Release video capture
if 'cap' in locals() and cap.isOpened():
    cap.release()
    print("Video resources released.")
else:
    print("No video resources to release.")

## Parameter Tuning Tips

If results look wrong, here are the most impactful parameters to adjust:

| Parameter | What to adjust | Symptoms of wrong value |
|---|---|---|
| `REAL_DISTANCE` | Re-measure calibration points | Scale too high/low affects all velocities proportionally |
| `CAL_P1`, `CAL_P2` | Use clearer reference points | Scale error if points are ambiguous |
| `START_FRAME` | Move earlier/later | Miss the serve if too late; noisy if too early |
| `BALL_POS` | Click the ball more carefully | Tracking starts from wrong position |
| `TEMPLATE_SIZE` | Increase if ball looks small; decrease if crowded | Tracking drift or mismatch |
| `SEARCH_RADIUS` | Increase if ball moves fast between frames | Ball goes out of search window |
| `SMOOTHING_WINDOW` | Increase to reduce jitter; decrease to preserve peaks | Over-smoothed or noisy |

**Workflow:** Adjust one parameter at a time, then re-run from Step 1 to see the effect.

## Limitations

Be honest with yourself about the accuracy of this tool:

- **Approximate velocity** from a single lateral view
- **Perspective error** increases with ball height above the calibration plane
- **Template matching** can drift or lose the ball in cluttered backgrounds
- **No spin estimation**, no 3D reconstruction, no magic
- **Calibration error** directly affects all velocity values

Use this as a **coarse sanity check**, not a precise measurement device.